In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder

# 0. CARGA Y PREPARACION DE DATOS
RUTA_CSV = "base de datos museos_5.csv"
df = pd.read_csv(RUTA_CSV, sep=";", encoding="latin-1")

# Variable objetivo: tipo de visitante (Nacionales / Extranjeros)
target_col = "NOM_TIPO"
le = LabelEncoder()
y = le.fit_transform(df[target_col])

# Candidatas a "features" para el modelo (todas numericas)
# y FECHA_CORTE / TIPO_cat_cod porque son constantes (no aportan info)
feature_cols = [
    "ANIO", "COD_DPTO", "COD_MES",
    "ADU_BOLESPPAGANTES", "EST_BOLESPPAGANTES", "NIN_BOLESPPAGANTES",
    "MIL_BOLESPPAGANTES", "ADM_BOLESPPAGANTES",
    "ADU_PAGANTES", "EST_PAGANTES", "NIN_PAGANTES", "TOTAL_PAGANTES",
    "ADU_BOLESPNOPAGANTES", "EST_BOLESPNOPAGANTES", "NIN_BOLESPNOPAGANTES",
    "MIL_BOLESPNOPAGANTES", "ADM_BOLESPNOPAGANTES",
    "ADU_NOPAGANTES", "EST_NOPAGANTES", "NIN_NOPAGANTES", "TOTAL_NOPAGANTES",
    "TOTAL",
]
X = df[feature_cols].values.astype(float)
N_FEATURES = X.shape[1]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f"Dataset: {df.shape[0]} filas, {N_FEATURES} features candidatas")
print(f"Target: '{target_col}' -> clases: {list(le.classes_)}")

# 1. REPRESENTACION DE LA PARTICULA (fuente de alimento)
# Cada fuente de alimento (particula) es un vector de N_FEATURES numeros
# Esta es la representacion continua-a-binaria estandar usada en ABC/PSO

def binarizar(posicion):
    return (posicion > 0.5).astype(int)


# 2. FUNCION DE APTITUD (FITNESS)
# Se entrena un arbol de decision usando SOLO las features seleccionadas y
# se mide su accuracy promedio por validacion cruzada (3-fold). El fitness

ALPHA_PENALIZACION = 0.02  #peso de la penalizacion por cantidad de features

def fitness(posicion):
    mascara = binarizar(posicion)
    n_sel = mascara.sum()

    if n_sel == 0:  #no se puede entrenar un modelo sin features
        return 0.0

    idx = np.where(mascara == 1)[0]
    clf = DecisionTreeClassifier(max_depth=6, random_state=42)
    scores = cross_val_score(clf, X_train[:, idx], y_train, cv=3, scoring="accuracy")
    accuracy = scores.mean()

    penalizacion = ALPHA_PENALIZACION * (n_sel / N_FEATURES)
    return accuracy - penalizacion


# PARAMETROS DEL ALGORITMO ABC
N_ABEJAS_EMPLEADAS = 8      # = numero de fuentes de alimento (particulas)
LIMIT = 5                   #intentos sin mejora antes de abandonar una fuente
MAX_ITER = 15                #criterio de parada por numero de ciclos
rng = np.random.default_rng(42)

# 3. INICIALIZACION DEL ENJAMBRE
poblacion = rng.uniform(0, 1, size=(N_ABEJAS_EMPLEADAS, N_FEATURES)) #el "array en numero entre 0 y 1 incluyendo decimales(fuente de alimetacion)"
fitness_pob = np.array([fitness(p) for p in poblacion])
trial = np.zeros(N_ABEJAS_EMPLEADAS, dtype=int)  # contador de "no mejora" por fuente

mejor_idx = np.argmax(fitness_pob)
mejor_pos = poblacion[mejor_idx].copy()
mejor_fit = fitness_pob[mejor_idx]

historial = [mejor_fit]
print("\n--- Enjambre inicial ---")
print(f"Mejor fitness inicial: {mejor_fit:.4f} "
      f"con {binarizar(mejor_pos).sum()} features")

# 4-5. COMPORTAMIENTO DE LA PARTICULA + EVOLUCION (ciclo principal de ABC)
for it in range(1, MAX_ITER + 1):

    #Fase de abejas EMPLEADAS
    # Cada abeja empleada explora un vecino de SU fuente, moviendose en direccion aleatoria hacia (o lejos de) otra fuente distinta k.
    for i in range(N_ABEJAS_EMPLEADAS):
        k = rng.integers(0, N_ABEJAS_EMPLEADAS)
        while k == i:
            k = rng.integers(0, N_ABEJAS_EMPLEADAS)
        phi = rng.uniform(-1, 1, size=N_FEATURES)
        candidato = poblacion[i] + phi * (poblacion[i] - poblacion[k])
        candidato = np.clip(candidato, 0, 1)

        f_cand = fitness(candidato)
        if f_cand > fitness_pob[i]:            # seleccion voraz (greedy)
            poblacion[i] = candidato
            fitness_pob[i] = f_cand
            trial[i] = 0
        else:
            trial[i] += 1

    #Fase de abejas OBSERVADORAS
    # Las observadoras eligen una fuente con probabilidad proporcional a su fitness (ruleta): las mejores fuentes reciben mas "visitas"
    prob = fitness_pob / (fitness_pob.sum() + 1e-9)
    for _ in range(N_ABEJAS_EMPLEADAS):
        i = rng.choice(N_ABEJAS_EMPLEADAS, p=prob)
        k = rng.integers(0, N_ABEJAS_EMPLEADAS)
        while k == i:
            k = rng.integers(0, N_ABEJAS_EMPLEADAS)
        phi = rng.uniform(-1, 1, size=N_FEATURES)
        candidato = poblacion[i] + phi * (poblacion[i] - poblacion[k])
        candidato = np.clip(candidato, 0, 1)

        f_cand = fitness(candidato)
        if f_cand > fitness_pob[i]:
            poblacion[i] = candidato
            fitness_pob[i] = f_cand
            trial[i] = 0
        else:
            trial[i] += 1

    #Fase de abejas EXPLORADORAS (scouts)
    #Si una fuente lleva "LIMIT" intentos sin mejorar, se considera agotada y se abandona: la abeja empleada se vuelve exploradora y genera una
    #fuente totalmente nueva al azar (evita quedar atrapado en un optimo local).
    for i in range(N_ABEJAS_EMPLEADAS):
        if trial[i] >= LIMIT:
            poblacion[i] = rng.uniform(0, 1, size=N_FEATURES)
            fitness_pob[i] = fitness(poblacion[i])
            trial[i] = 0

    #Actualizar la mejor solucion global
    it_mejor_idx = np.argmax(fitness_pob)
    if fitness_pob[it_mejor_idx] > mejor_fit:
        mejor_fit = fitness_pob[it_mejor_idx]
        mejor_pos = poblacion[it_mejor_idx].copy()

    historial.append(mejor_fit)
    print(f"Iteracion {it:02d}/{MAX_ITER} -> mejor fitness = {mejor_fit:.4f} "
          f"({binarizar(mejor_pos).sum()} features seleccionadas)")

# 6. FINALIZACION: mejor solucion encontrada y evaluacion final
mascara_final = binarizar(mejor_pos)
features_elegidas = [feature_cols[i] for i in range(N_FEATURES) if mascara_final[i] == 1]
idx_final = np.where(mascara_final == 1)[0]

clf_final = DecisionTreeClassifier(max_depth=6, random_state=42)
clf_final.fit(X_train[:, idx_final], y_train)
acc_test = clf_final.score(X_test[:, idx_final], y_test)

print("\n=================== RESULTADO FINAL (ABC) ===================")
print(f"Fitness (train, CV) de la mejor solucion: {mejor_fit:.4f}")
print(f"Numero de features seleccionadas: {len(features_elegidas)} de {N_FEATURES}")
print(f"Features elegidas: {features_elegidas}")
print(f"Accuracy en el conjunto de TEST con esas features: {acc_test:.4f}")

# Comparacion: modelo con TODAS las features (baseline)
clf_todas = DecisionTreeClassifier(max_depth=6, random_state=42)
clf_todas.fit(X_train, y_train)
acc_todas = clf_todas.score(X_test, y_test)
print(f"\nBaseline (usando TODAS las {N_FEATURES} features): accuracy = {acc_todas:.4f}")
print(f"ABC logro un accuracy similar/mejor usando solo "
      f"{len(features_elegidas)} features -> modelo mas simple y eficiente.")

Dataset: 20524 filas, 22 features candidatas
Target: 'NOM_TIPO' -> clases: ['Extranjeros', 'Nacionales']

--- Enjambre inicial ---
Mejor fitness inicial: 0.8594 con 13 features
Iteracion 01/15 -> mejor fitness = 0.8599 (12 features seleccionadas)
Iteracion 02/15 -> mejor fitness = 0.8599 (12 features seleccionadas)
Iteracion 03/15 -> mejor fitness = 0.8606 (12 features seleccionadas)
Iteracion 04/15 -> mejor fitness = 0.8630 (9 features seleccionadas)
Iteracion 05/15 -> mejor fitness = 0.8630 (9 features seleccionadas)
Iteracion 06/15 -> mejor fitness = 0.8630 (9 features seleccionadas)
Iteracion 07/15 -> mejor fitness = 0.8630 (9 features seleccionadas)
Iteracion 08/15 -> mejor fitness = 0.8630 (9 features seleccionadas)
Iteracion 09/15 -> mejor fitness = 0.8630 (9 features seleccionadas)
Iteracion 10/15 -> mejor fitness = 0.8630 (9 features seleccionadas)
Iteracion 11/15 -> mejor fitness = 0.8640 (8 features seleccionadas)
Iteracion 12/15 -> mejor fitness = 0.8640 (8 features selecci